In [ ]:
import os
import re
import joblib
import numpy as np
import pandas as pd

from collections import Counter
from typing import List
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import StandardScaler

In [ ]:
class TextPreprocessor(BaseEstimator, TransformerMixin):

    def clean_text(self, text: str) -> str:
        text = text.lower()
        text = re.sub(r"http\S+", "", text)
        text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return [self.clean_text(text) for text in X]

In [ ]:
class StatisticalFeatures(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        features = []
        for text in X:
            features.append([
                len(text),
                text.count("!"),
                text.count("?"),
                sum(word.isupper() for word in text.split())
            ])
        return np.array(features)

In [ ]:
def build_dataset():
    texts = [
        "Invoice not generated",
        "Refund not received",
        "Incorrect billing amount",
        "Duplicate charge",
        "Subscription payment failed",
        "Billing issue urgent",
        "Server is down",
        "Application crashes on login",
        "Database timeout error",
        "System failure ASAP",
        "API not responding",
        "Bug in production environment",
        "Need contract review",
        "Legal compliance question",
        "NDA document request",
        "Data privacy concern",
        "Terms clarification",
        "Agreement amendment required"
    ]

    labels = [
        "Billing","Billing","Billing","Billing","Billing","Billing",
        "Technical","Technical","Technical","Technical","Technical","Technical",
        "Legal","Legal","Legal","Legal","Legal","Legal"
    ]

    return pd.DataFrame({"text": texts, "label": labels})


df = build_dataset()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["text"],
    df["label"],
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

In [ ]:
class_counts = Counter(y_train)
min_class_size = min(class_counts.values())

n_splits = min(5, min_class_size)

print("Using n_splits =", n_splits)

In [ ]:
word_tfidf = TfidfVectorizer(
    ngram_range=(1,2),
    max_features=2000,
    stop_words="english"
)

char_tfidf = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3,5),
    max_features=1000
)

feature_union = FeatureUnion([
    ("word_features", word_tfidf),
    ("char_features", char_tfidf),
    ("stat_features", StatisticalFeatures())
])

In [ ]:
base_svm = LinearSVC(class_weight="balanced")

In [ ]:
advanced_pipeline = Pipeline([
    ("preprocess", TextPreprocessor()),
    ("features", feature_union),
    ("scaler", StandardScaler(with_mean=False)),
    ("classifier", base_svm)
])

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    advanced_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="f1_macro"
)

print("CV F1:", cv_scores)
print("Mean:", np.mean(cv_scores))

In [ ]:
advanced_pipeline.fit(X_train, y_train)

In [ ]:
calibrated_model = CalibratedClassifierCV(
    advanced_pipeline,
    cv="prefit",
    method="sigmoid"
)

calibrated_model.fit(X_train, y_train)

In [ ]:
os.makedirs("model", exist_ok=True)

joblib.dump(
    advanced_pipeline,
    "model/mvr_classifier.joblib"
)

print("Baseline Classifier model saved.")

In [ ]:
class BaselineClassifier:
    
    def __init__(self, model_path: str):
        self.model = joblib.load(model_path)
    
    def predict(self, text: str):
        return self.model.predict([text])[0]
    
    def predict_proba(self, text: str):
        probs = self.model.predict_proba([text])[0]
        classes = self.model.classes_
        return dict(zip(classes, probs))